In [ ]:
!pip install -q pygments

In [ ]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get -y update
!apt-get install -y google-chrome-stable
!pip install selenium beautifulsoup4

OK
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 https://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:4 https://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,213 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [39.2 kB]
Fetched 60.3 kB in 2s (36.1 kB/s)
Reading package lists... Done
W: Target Packages (main/binary-amd64/Packages) is configured multiple times in

In [ ]:
!pip install undetected-chromedriver

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time

def crawl_exrx_parent_categories_dynamic(index_url):
    print(f"🕸️ Dynamic Parent Spider starting on: {index_url}")

    # ==========================================
    # 1. SETUP SELENIUM
    # ==========================================
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

    try:
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(index_url)
        time.sleep(3)
        html = driver.page_source
    except Exception as e:
        print(f"Error loading index page: {e}")
        return []
    finally:
        if 'driver' in locals():
            driver.quit()

    # ==========================================
    # 2. PARSE USING DOM STRUCTURE
    # ==========================================
    soup = BeautifulSoup(html, 'html.parser')
    article = soup.find('article')

    if not article:
        print("Could not find the main article container.")
        return []

    category_links = []

    # Find only the TOP-LEVEL lists (lists that are not nested inside another list item)
    top_uls = [ul for ul in article.find_all('ul') if ul.parent.name != 'li']

    for ul in top_uls:
        for li in ul.find_all('li', recursive=False):

            # Scenario A: The list item has a direct, clickable link (e.g., Neck)
            direct_a = li.find('a', recursive=False)

            if direct_a and direct_a.has_attr('href'):
                href = direct_a['href']
                if '#' not in href: # Filter out the child muscles
                    full_url = urljoin(index_url, href)
                    category_links.append({
                        'category': direct_a.get_text(strip=True),
                        'url': full_url
                    })

            # Scenario B: The list item is NOT clickable (e.g., "Other Exercises")
            # So, we look for a nested list inside of it.
            elif not direct_a:
                nested_ul = li.find('ul', recursive=False)
                if nested_ul:
                    # Loop through the children of this nested list (Plyometrics, Cardio, etc.)
                    for nested_li in nested_ul.find_all('li', recursive=False):
                        nested_a = nested_li.find('a', recursive=False)

                        if nested_a and nested_a.has_attr('href'):
                            href = nested_a['href']
                            if '#' not in href:
                                full_url = urljoin(index_url, href)
                                category_links.append({
                                    'category': nested_a.get_text(strip=True),
                                    'url': full_url
                                })

    # Remove duplicates
    unique_categories = []
    seen = set()
    for item in category_links:
        if item['url'] not in seen:
            unique_categories.append(item)
            seen.add(item['url'])

    return unique_categories

# ==========================================
# Run the Spider
# ==========================================
target_index = 'https://exrx.net/Lists/Directory'
parent_categories = crawl_exrx_parent_categories_dynamic(target_index)

if parent_categories:
    print(f"\n✅ Successfully crawled {len(parent_categories)} Parent Categories!\n")

    df_urls = pd.DataFrame(parent_categories)
    print(df_urls.head(25).to_string(index=False))

    df_urls.to_csv('/content/parent_categories.csv', index=False)
    print("\n💾 Saved to '/content/parent_categories.csv'")
else:
    print("No items found. Check the HTML structure or URL.")

🕸️ Dynamic Parent Spider starting on: https://exrx.net/Lists/Directory

✅ Successfully crawled 15 Parent Categories!

                 category                                         url
                     Neck        https://exrx.net/Lists/ExList/NeckWt
                Shoulders      https://exrx.net/Lists/ExList/ShouldWt
               Upper Arms         https://exrx.net/Lists/ExList/ArmWt
                 Forearms     https://exrx.net/Lists/ExList/ForeArmWt
                     Back        https://exrx.net/Lists/ExList/BackWt
                    Chest       https://exrx.net/Lists/ExList/ChestWt
                    Waist       https://exrx.net/Lists/ExList/WaistWt
                     Hips        https://exrx.net/Lists/ExList/HipsWt
                   Thighs       https://exrx.net/Lists/ExList/ThighWt
                   Calves        https://exrx.net/Lists/ExList/CalfWt
Olympic-style Weightlifts https://exrx.net/Lists/OlympicWeightlifting
              Plyometrics       https://ex

In [ ]:
df_urls.head()

,category,url
0,Neck,https://exrx.net/Lists/ExList/NeckWt
1,Shoulders,https://exrx.net/Lists/ExList/ShouldWt
2,Upper Arms,https://exrx.net/Lists/ExList/ArmWt
3,Forearms,https://exrx.net/Lists/ExList/ForeArmWt
4,Back,https://exrx.net/Lists/ExList/BackWt


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time


def crawl_exrx_index_breadcrumbs(index_url):
    print(f"🕸️ Recursive Spider starting on: {index_url}")

    # ==========================================
    # 1. SETUP SELENIUM (OFFICIAL CHROME MODE)
    # ==========================================
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    )

    try:
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(index_url)
        time.sleep(3)
        html = driver.page_source
    except Exception as e:
        print(f"Error loading index page: {e}")
        return []
    finally:
        if 'driver' in locals():
            driver.quit()

    # ==========================================
    # 2. PARSE WITH RECURSIVE BREADCRUMBS
    # ==========================================
    soup = BeautifulSoup(html, 'html.parser')
    article = soup.find('article')

    if not article:
        print("Could not find the main article container.")
        return []

    exercise_links = []

    valid_exercise_paths = [
        'WeightExercises',
        'BodyWeight',
        'Plyometrics',
        'Kettlebell',
        'Stretches',
        'OlympicWeightlifting'
    ]

    # --- THE RECURSIVE ENGINE ---
    def traverse_ul(ul_node, current_path):

        for li in ul_node.find_all('li', recursive=False):

            li_texts = []
            valid_a_tags = []  # ✅ FIX: Now an array to hold multiple links!

            for child in li.contents:

                if child.name == 'ul':
                    continue

                # Check if it's a valid exercise link
                if child.name == 'a' and child.has_attr('href'):
                    if any(p in child['href'] for p in valid_exercise_paths):
                        valid_a_tags.append(child)  # ✅ Save ALL links on this line

                # Grab the text safely (ignoring the random slashes ExRx uses)
                if isinstance(child, str):
                    text = child.strip()
                    if text and text not in ['/', ',', '&', '-']:
                        li_texts.append(text)

                elif child.name and child.name != 'a':
                    text = child.get_text(strip=True)
                    if text:
                        li_texts.append(text)

            # Combine the text to create the base label for this bullet point
            base_label = " ".join(li_texts).replace("\u00a0", " ").strip()

            # This is the path we pass down if there are nested lists inside this bullet
            new_path = current_path + [base_label] if base_label else current_path

            # ✅ FIX: Loop through ALL the links we found on this single line
            for a_tag in valid_a_tags:
                href = a_tag['href']
                full_url = urljoin(index_url, href)
                link_text = a_tag.get_text(strip=True)

                # If there are multiple links on one line, attach the link text
                # to the breadcrumb to tell them apart!
                if len(valid_a_tags) > 1:
                    final_path = new_path + [link_text]
                elif not base_label and link_text:
                    final_path = current_path + [link_text]
                else:
                    final_path = new_path

                breadcrumb = " > ".join(final_path)

                # Avoid duplicates
                if not any(ex['url'] == full_url for ex in exercise_links):
                    exercise_links.append({
                        'exercise_name': breadcrumb,
                        'url': full_url
                    })

            # If there is a nested <ul> inside this <li>, dive into it!
            for child_ul in li.find_all('ul', recursive=False):
                traverse_ul(child_ul, new_path)

    # --- START THE CRAWL ---
    all_uls = article.find_all('ul')

    for ul in all_uls:

        if ul.parent.name == 'li':
            continue

        h2_tag = ul.find_previous('h2')
        base_category = h2_tag.get_text(strip=True) if h2_tag else "Uncategorized"

        traverse_ul(ul, [base_category])

    return exercise_links


# ==========================================
# Run the Spider
# ==========================================
target_index = 'https://exrx.net/Lists/CardioExercises'

all_back_exercises = crawl_exrx_index_breadcrumbs(target_index)

if all_back_exercises:
    print(f"\n✅ Successfully crawled {len(all_back_exercises)} Back Exercises!\n")

    df_urls = pd.DataFrame(all_back_exercises)

    print(df_urls.head(10).to_string(index=False))

    df_urls.to_csv('/content/back_exercises_urls.csv', index=False)
    print("\n💾 Saved to '/content/back_exercises_urls.csv'")
else:
    print("No exercises found. Check the HTML structure or URL.")

🕸️ Recursive Spider starting on: https://exrx.net/Lists/CardioExercises

✅ Successfully crawled 3 Back Exercises!

                            exercise_name                                         url
           Uncategorized > Burpee Push-up   https://exrx.net/Plyometrics/BurpeePushup
               Uncategorized > Kettlebell  https://exrx.net/Lists/KettlebellExercises
Uncategorized > Olympic-style Weightlifts https://exrx.net/Lists/OlympicWeightlifting

💾 Saved to '/content/back_exercises_urls.csv'


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time
import random
from tqdm import tqdm
import gc # Python's Garbage Collector to force memory cleanup

# ==========================================
# 1. THE MEMORY-SAFE FUNCTION
# ==========================================
def crawl_exrx_category_safe(index_url):
    html = ""
    driver = None

    try:
        chrome_options = Options()
        chrome_options.add_argument("--headless=new")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

        driver = webdriver.Chrome(options=chrome_options)
        driver.get(index_url)
        time.sleep(3)

        if "Just a moment" in driver.title or "Attention Required" in driver.title:
            print(f"\n   ⏳ Cloudflare check on {index_url}. Waiting 8 seconds...")
            time.sleep(8)

        html = driver.page_source

    except Exception as e:
        print(f"\n   ❌ Error loading {index_url}: {e}")
        return []

    finally:
        if driver:
            driver.quit()

    if not html: return []

    # ==========================================
    # 2. PARSE WITH BEAUTIFULSOUP
    # ==========================================
    soup = BeautifulSoup(html, 'html.parser')

    container = soup.find('article') or soup.find('main') or soup.find('div', class_='container') or soup.body

    if not container:
        print(f"\n   ⚠️ Could not find container on {index_url}")
        return []

    exercise_links = []

    # ✨ FIX 1: EXPANDED PATHS to catch Cardio and Miscellaneous exercises
    valid_paths = [
        'weightexercises', 'bodyweight', 'plyometrics', 'powerexercises',
        'kettlebell', 'kettlebellexercises', 'stretches', 'olympicweightlifting',
        'cardio', 'other'
    ]

    def traverse_ul(ul_node, current_path):
        for li in ul_node.find_all('li', recursive=False):
            li_texts = []
            valid_a_tags = []

            for child in li.contents:
                if child.name == 'ul': continue

                if child.name == 'a' and child.has_attr('href'):
                    href_lower = child['href'].lower()
                    if any(p in href_lower for p in valid_paths):
                        valid_a_tags.append(child)

                if isinstance(child, str):
                    text = child.strip()
                    if text and text not in ['/', ',', '&', '-']:
                        li_texts.append(text)
                elif child.name and child.name != 'a':
                    text = child.get_text(strip=True)
                    if text: li_texts.append(text)

            base_label = " ".join(li_texts).replace("\u00a0", " ").strip()
            new_path = current_path + [base_label] if base_label else current_path

            for a_tag in valid_a_tags:
                href = a_tag['href']
                full_url = urljoin(index_url, href)
                link_text = a_tag.get_text(strip=True)

                if len(valid_a_tags) > 1:
                    final_path = new_path + [link_text]
                elif not base_label and link_text:
                    final_path = current_path + [link_text]
                else:
                    final_path = new_path

                breadcrumb = " > ".join(final_path)

                # We still avoid duplicates WITHIN the same page
                if not any(ex['url'] == full_url for ex in exercise_links):
                    exercise_links.append({
                        'exercise_name': breadcrumb,
                        'url': full_url
                    })

            for child_ul in li.find_all('ul', recursive=False):
                traverse_ul(child_ul, new_path)

    all_uls = container.find_all('ul')
    for ul in all_uls:
        if ul.parent.name == 'li': continue

        h2_tag = ul.find_previous(['h2', 'h3', 'h4'])
        base_category = h2_tag.get_text(strip=True) if h2_tag else "General"

        traverse_ul(ul, [base_category])

    return exercise_links

# ==========================================
# 3. THE MASTER LOOP
# ==========================================

try:
    df_parents = pd.read_csv('/content/parent_categories.csv')
    print(f"Loaded {len(df_parents)} Parent Categories.")
except FileNotFoundError:
    print("Could not find parent_categories.csv!")
    exit()

master_exercise_list = []

print("\n🚀 Beginning Memory-Safe Master Crawl...\n")

for index, row in tqdm(df_parents.iterrows(), total=len(df_parents), desc="Crawling"):
    parent_url = row['url']
    parent_category_name = row['category']

    scraped_links = crawl_exrx_category_safe(parent_url)

    for link in scraped_links:
        if not link['exercise_name'].startswith(parent_category_name):
            link['exercise_name'] = f"{parent_category_name} > {link['exercise_name']}"
        link['parent_category'] = parent_category_name

    master_exercise_list.extend(scraped_links)

    if len(scraped_links) == 0:
        print(f"\n   ⚠️ WARNING: 0 links found for {parent_category_name}.")
    else:
        print(f"\n   ✔️ {parent_category_name}: Found {len(scraped_links)} exercises")

    time.sleep(random.uniform(3.0, 6.0))
    gc.collect()

# ==========================================
# 4. SAVE AND VERIFY
# ==========================================
if master_exercise_list:
    df_master = pd.DataFrame(master_exercise_list)

    # ✨ FIX 2: DISABLED GLOBAL DEDUPLICATION
    # By commenting this out, exercises listed in multiple categories (e.g., Back AND Shoulders) are kept!
    # df_master = df_master.drop_duplicates(subset=['url'])

    print("\n" + "="*40)
    print(" 📊 FINAL DATABASE SUMMARY")
    print("="*40)
    print(f"Total Exercises Captured: {len(df_master)}\n")
    print(df_master['parent_category'].value_counts().to_string())
    print("="*40)

    # We will keep the 'parent_category' column in the final CSV just in case you need to filter by it later
    df_master.to_csv('/content/all_exercises_master_list.csv', index=False)
    print("\n💾 Saved successfully to '/content/all_exercises_master_list.csv'")
else:
    print("No exercises found.")

Loaded 15 Parent Categories.

🚀 Beginning Memory-Safe Master Crawl...



Crawling:   0%|          | 0/15 [00:00<?, ?it/s]


   ✔️ Neck: Found 19 exercises


Crawling:   7%|▋         | 1/15 [00:15<03:37, 15.56s/it]


   ✔️ Shoulders: Found 57 exercises


...


   ✔️ Kettlebell: Found 15 exercises


Crawling:  93%|█████████▎| 14/15 [03:35<00:18, 18.81s/it]


   ✔️ Miscellaneous: Found 33 exercises


Crawling: 100%|██████████| 15/15 [03:47<00:00, 15.19s/it]


 📊 FINAL DATABASE SUMMARY
Total Exercises Captured: 873

parent_category
Hips                         133
Thighs                       117
Waist                        106
Back                          87
Plyometrics                   74
Calves                        58
Chest                         58
Upper Arms                    57
Shoulders                     57
Miscellaneous                 33
Olympic-style Weightlifts     29
Forearms                      25
Neck                          19
Kettlebell                    15
Cardio & Conditioning          5

💾 Saved successfully to '/content/all_exercises_master_list.csv'


In [ ]:
print(df_master.shape)
mask=df_master["exercise_name"].str.contains("Cardio & Conditioning")
df_master[mask]

(873, 3)


,exercise_name,url,parent_category
742,Olympic-style Weightlifts > Other Exercises > ...,https://exrx.net/Lists/CardioExercises,Olympic-style Weightlifts
816,Plyometrics > Other Exercises > Cardio & Condi...,https://exrx.net/Lists/CardioExercises,Plyometrics
820,Cardio & Conditioning > Calisthenics > Burpee ...,https://exrx.net/Plyometrics/BurpeePushup,Cardio & Conditioning
821,Cardio & Conditioning > Other Exercises > Kett...,https://exrx.net/Lists/KettlebellExercises,Cardio & Conditioning
822,Cardio & Conditioning > Other Exercises > Olym...,https://exrx.net/Lists/OlympicWeightlifting,Cardio & Conditioning
823,Cardio & Conditioning > Other Exercises > Plyo...,https://exrx.net/Lists/PowerExercises,Cardio & Conditioning
824,Cardio & Conditioning > Other Exercises > Misc...,https://exrx.net/Lists/OtherExercises,Cardio & Conditioning
836,Kettlebell > Other Exercises > Cardio & Condit...,https://exrx.net/Lists/CardioExercises,Kettlebell
869,Miscellaneous > Other Exercises > Cardio & Con...,https://exrx.net/Lists/CardioExercises,Miscellaneous


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
import random
from tqdm import tqdm
import gc
import os

# ==========================================
# 1. MEMORY-SAFE HTML FETCHER
# ==========================================
def fetch_html_safe(url):
    html = ""
    driver = None
    try:
        chrome_options = Options()
        chrome_options.add_argument("--headless=new")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

        driver = webdriver.Chrome(options=chrome_options)
        driver.get(url)
        time.sleep(3)

        # Cloudflare Check
        if "Just a moment" in driver.title or "Attention Required" in driver.title:
            time.sleep(8)

        html = driver.page_source
    except Exception as e:
        print(f"\n❌ Error fetching {url}: {e}")
    finally:
        if driver:
            driver.quit()

    return html

# ==========================================
# 2. THE PERFECTED PARSER
# ==========================================
def parse_exercise_data(html, url, breadcrumb, parent_category):
    soup = BeautifulSoup(html, 'html.parser')

    # Initialize with our critical context!
    exercise_data = {
        'breadcrumb_path': breadcrumb,
        'parent_category': parent_category,
        'source_url': url
    }

    # --- A. Get the Title ---
    title_tag = soup.find('h1', class_='page-title') or soup.find('h1')
    exercise_data['exercise_name'] = title_tag.text.strip() if title_tag else "Unknown"

    article = soup.find('article') or soup.find('main') or soup.find('div', class_='container')
    if not article:
        return exercise_data

    # --- B. Get the Classification Table ---
    class_table = article.find('table')
    if class_table:
        for row in class_table.find_all('tr'):
            columns = row.find_all('td')
            if len(columns) == 2:
                key = columns[0].text.strip().replace(':', '').replace(' ', '_').lower()
                value = columns[1].text.strip()
                exercise_data[key] = value

    # --- C. Get Instructions ---
    instructions = {}
    instr_header = article.find(lambda tag: tag.name in ['h2', 'h3'] and 'Instructions' in tag.text)

    if instr_header:
        current_subheading = None
        for sibling in instr_header.find_next_siblings():
            if sibling.name in ['h2', 'h3'] and sibling.text.strip() not in ['Preparation', 'Execution']:
                break

            if sibling.name == 'strong' or (sibling.name == 'p' and sibling.find('strong')):
                current_subheading = sibling.text.strip().lower()
                instructions[current_subheading] = ""
            elif sibling.name == 'p' and current_subheading:
                instructions[current_subheading] += sibling.text.strip() + " "

        exercise_data['instructions'] = instructions

    # --- D. Get Extra Sections ---
    extra_sections = ['Comments', 'Easier', 'Harder']
    for section_name in extra_sections:
        target_tag = article.find(lambda tag:
            (tag.name in ['h2', 'h3', 'h4'] and section_name in tag.text) or
            (tag.name == 'p' and tag.find('strong') and section_name in tag.text)
        )

        if target_tag:
            content = []
            for sibling in target_tag.find_next_siblings():
                if sibling.name in ['h2', 'h3', 'h4']: break
                if sibling.name == 'p' and sibling.find('strong'):
                    check_text = sibling.text.strip()
                    if any(s in check_text for s in extra_sections + ['Muscles', 'Force', 'Preparation', 'Execution']): break

                if sibling.name == 'p' and not sibling.find('strong'):
                    text = sibling.text.strip()
                    if text: content.append(text)
                elif sibling.name == 'ul':
                    for li in sibling.find_all('li'):
                        clean_text = li.contents[0].text.strip() if li.contents else li.text.strip()
                        if clean_text: content.append(f"• {clean_text}")

            if content:
                exercise_data[section_name.lower()] = "\n".join(content)

    # --- E. Get Muscles ---
    muscles = {}
    muscle_header = article.find(lambda tag: tag.name in ['h2', 'h3'] and 'Muscles' in tag.text)

    if muscle_header:
        current_muscle_group = None
        for sibling in muscle_header.find_next_siblings():
            if sibling.name in ['h2', 'h3']: break

            if sibling.name == 'p' and sibling.find('strong'):
                current_muscle_group = sibling.text.strip().lower().replace(' ', '_')
                muscles[current_muscle_group] = []
            elif sibling.name == 'ul' and current_muscle_group:
                for li in sibling.find_all('li'):
                    clean_text = li.contents[0].text.strip() if li.contents else li.text.strip()
                    if clean_text: muscles[current_muscle_group].append(clean_text)

        if muscles:
            exercise_data['muscles'] = muscles

    return exercise_data

# ==========================================
# 3. THE MASTER SCRAPE LOOP
# ==========================================
try:
    df_master = pd.read_csv('/content/all_exercises_master_list.csv')
except FileNotFoundError:
    print("Could not find master list! Ensure 'all_exercises_master_list.csv' is in your folder.")
    exit()

output_file = '/content/final_exrx_database.json'
scraped_data_list = []

# If we crashed previously, load the existing data to resume!
if os.path.exists(output_file):
    with open(output_file, 'r', encoding='utf-8') as f:
        scraped_data_list = json.load(f)
    print(f"Resuming from checkpoint! {len(scraped_data_list)} exercises already scraped.")

# Get a list of URLs we have already scraped to avoid double-work
completed_urls = [item.get('source_url') for item in scraped_data_list]

print(f"\n🚀 Beginning Deep Content Scrape for {len(df_master)} exercises...\n")

for index, row in tqdm(df_master.iterrows(), total=len(df_master), desc="Scraping Exercises"):
    url = row['url']
    breadcrumb = row['exercise_name']

    # We use .get() here safely in case 'parent_category' was dropped in previous steps
    parent = row.get('parent_category', breadcrumb.split(' > ')[0])

    if url in completed_urls:
        continue # Skip if we already scraped it in a previous run

    html = fetch_html_safe(url)

    if html:
        data = parse_exercise_data(html, url, breadcrumb, parent)
        scraped_data_list.append(data)

    # Random human delay
    time.sleep(random.uniform(2.0, 4.0))
    gc.collect() # Free up RAM

    # ✨ THE CHECKPOINT SYSTEM ✨
    # Save to JSON every 25 exercises so we never lose data!
    if (index + 1) % 25 == 0:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(scraped_data_list, f, indent=4, ensure_ascii=False)
        print(f"\n   💾 Checkpoint Saved: {len(scraped_data_list)} exercises secured.")

# ==========================================
# 4. FINAL SAVE
# ==========================================
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(scraped_data_list, f, indent=4, ensure_ascii=False)

print("\n" + "="*50)
print(" 🎉 SCRAPING COMPLETE! 🎉")
print("="*50)
print(f" Total Exercises Extracted: {len(scraped_data_list)}")
print(f" Database Saved To: {output_file}")


🚀 Beginning Deep Content Scrape for 873 exercises...



Scraping Exercises:   3%|▎         | 25/873 [07:32<5:07:23, 21.75s/it]


   💾 Checkpoint Saved: 25 exercises secured.


Scraping Exercises:   6%|▌         | 50/873 [15:20<3:50:19, 16.79s/it]


   💾 Checkpoint Saved: 50 exercises secured.


...


   💾 Checkpoint Saved: 825 exercises secured.


Scraping Exercises:  97%|█████████▋| 850/873 [4:44:00<06:33, 17.10s/it]


   💾 Checkpoint Saved: 850 exercises secured.


Scraping Exercises: 100%|██████████| 873/873 [4:51:08<00:00, 20.01s/it]


 🎉 SCRAPING COMPLETE! 🎉
 Total Exercises Extracted: 873
 Database Saved To: /content/final_exrx_database.json


In [ ]:
import shutil
import os

# Create destination directory if it doesn't exist
destination_dir = '/content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises/scrap/'
os.makedirs(destination_dir, exist_ok=True)

# Source and destination files
source_file = '/content/final_exrx_database.json'
destination = os.path.join(destination_dir, 'final_exrx_database.json')

# Copy the file
shutil.copy(source_file, destination)
print(f"File copied to {destination}")

File copied to /content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises/scrap/final_exrx_database.json


In [ ]:
import json
import random
from IPython.display import display, HTML

# 1. Load your newly minted database
file_path = '/content/final_exrx_database.json'
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        database = json.load(f)
    print(f" Successfully loaded {len(database)} exercises from database.\n")
except FileNotFoundError:
    print(" Could not find the JSON file. Check the path!")
    exit()

# 2. Enhanced CSS styling
css_styles = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

.exercise-container {
    font-family: 'Inter', sans-serif;
    max-width: 900px;
    margin: 20px auto;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 3px;
    border-radius: 20px;
    box-shadow: 0 20px 40px rgba(0,0,0,0.3);
    animation: slideIn 0.5s ease-out;
}

@keyframes slideIn {
    from {
        opacity: 0;
        transform: translateY(30px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}

.exercise-card {
    background: white;
    border-radius: 18px;
    padding: 25px;
    position: relative;
    overflow: hidden;
}

.exercise-card::before {
    content: '';
    position: absolute;
    top: 0;
    left: 0;
    right: 0;
    height: 6px;
    background: linear-gradient(90deg, #ff6b6b, #4ecdc4, #45b7d1, #96ceb4);
    animation: gradientShift 3s ease infinite;
    background-size: 300% 300%;
}

@keyframes gradientShift {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

.exercise-header {
    display: flex;
    align-items: center;
    margin-bottom: 20px;
    border-bottom: 2px solid #f0f0f0;
    padding-bottom: 15px;
}

.exercise-number {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
    width: 40px;
    height: 40px;
    border-radius: 12px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: bold;
    font-size: 1.2em;
    margin-right: 15px;
    transform: rotate(-5deg);
    box-shadow: 0 4px 10px rgba(102, 126, 234, 0.4);
}

.exercise-title {
    font-size: 1.8em;
    font-weight: 700;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin: 0;
    flex: 1;
    text-transform: uppercase;
    letter-spacing: 1px;
}

.badge {
    display: inline-block;
    padding: 5px 12px;
    border-radius: 20px;
    font-size: 0.8em;
    font-weight: 600;
    margin-right: 8px;
    margin-bottom: 8px;
    animation: fadeIn 0.5s ease-out;
}

@keyframes fadeIn {
    from { opacity: 0; transform: scale(0.9); }
    to { opacity: 1; transform: scale(1); }
}

.badge-primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; }
.badge-success { background: linear-gradient(135deg, #84fab0 0%, #8fd3f4 100%); color: #333; }
.badge-warning { background: linear-gradient(135deg, #fad0c4 0%, #ffd1ff 100%); color: #333; }
.badge-info { background: linear-gradient(135deg, #a1c4fd 0%, #c2e9fb 100%); color: #333; }

.info-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
    gap: 15px;
    margin: 20px 0;
    padding: 15px;
    background: #f8f9fa;
    border-radius: 15px;
}

.info-item {
    display: flex;
    flex-direction: column;
}

.info-label {
    font-size: 0.8em;
    color: #666;
    text-transform: uppercase;
    letter-spacing: 1px;
    margin-bottom: 5px;
}

.info-value {
    font-weight: 600;
    color: #333;
    font-size: 1.1em;
}

.muscle-group {
    margin: 15px 0;
    padding: 15px;
    background: #f8f9fa;
    border-radius: 15px;
    border-left: 5px solid #667eea;
    transition: transform 0.2s;
}

.muscle-group:hover {
    transform: translateX(5px);
}

.muscle-group-title {
    font-weight: 700;
    color: #667eea;
    margin-bottom: 10px;
    text-transform: uppercase;
    font-size: 0.9em;
    letter-spacing: 1px;
}

.muscle-tags {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
}

.muscle-tag {
    background: white;
    padding: 4px 12px;
    border-radius: 15px;
    font-size: 0.9em;
    border: 1px solid #e0e0e0;
    box-shadow: 0 2px 4px rgba(0,0,0,0.05);
    transition: all 0.2s;
}

.muscle-tag:hover {
    background: #667eea;
    color: white;
    border-color: #667eea;
    transform: translateY(-2px);
}

.instruction-step {
    margin: 15px 0;
    padding: 15px;
    background: linear-gradient(135deg, #f5f7fa 0%, #f8f9fa 100%);
    border-radius: 12px;
    border: 1px solid #e0e0e0;
    transition: all 0.2s;
}

.instruction-step:hover {
    box-shadow: 0 5px 15px rgba(0,0,0,0.1);
    transform: translateY(-2px);
}

.instruction-title {
    font-weight: 700;
    color: #667eea;
    text-transform: capitalize;
    margin-bottom: 8px;
    display: flex;
    align-items: center;
}

.instruction-title::before {
    content: '→';
    margin-right: 8px;
    color: #764ba2;
    font-weight: bold;
}

.instruction-text {
    color: #444;
    line-height: 1.6;
    font-size: 0.95em;
}

/* Enhanced Extra Notes Styling */
.extra-notes-section {
    margin-top: 25px;
    background: linear-gradient(135deg, #fff8f8 0%, #fff3f3 100%);
    border-radius: 15px;
    padding: 20px;
    border: 2px solid #ff6b6b;
    box-shadow: 0 10px 30px rgba(255, 107, 107, 0.2);
}

.extra-notes-title {
    font-size: 1.3em;
    font-weight: 700;
    color: #d63031;
    margin-bottom: 15px;
    display: flex;
    align-items: center;
    gap: 10px;
    border-bottom: 2px solid #ff6b6b;
    padding-bottom: 10px;
}

.extra-notes-title::before {
    content: '';
    font-size: 1.2em;
}

.extra-note-card {
    background: white;
    border-radius: 12px;
    margin-bottom: 15px;
    padding: 15px;
    border-left: 8px solid;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
    transition: all 0.3s ease;
    animation: slideInNote 0.5s ease-out;
}

.extra-note-card:hover {
    transform: translateX(5px);
    box-shadow: 0 8px 25px rgba(0,0,0,0.1);
}

@keyframes slideInNote {
    from {
        opacity: 0;
        transform: translateX(-20px);
    }
    to {
        opacity: 1;
        transform: translateX(0);
    }
}

.note-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    margin-bottom: 12px;
    padding-bottom: 8px;
    border-bottom: 1px dashed #e0e0e0;
}

.note-title {
    font-weight: 700;
    font-size: 1.1em;
    display: flex;
    align-items: center;
    gap: 8px;
}

.note-title.comments { color: #0984e3; }
.note-title.easier { color: #00b894; }
.note-title.harder { color: #d63031; }

.note-badge {
    background: #f0f0f0;
    padding: 4px 10px;
    border-radius: 20px;
    font-size: 0.8em;
    font-weight: 600;
    color: #666;
}

.note-content {
    color: #2d3436;
    line-height: 1.7;
    font-size: 0.95em;
    padding: 5px 0;
}

.note-content p {
    margin: 0;
}

.note-footer {
    margin-top: 10px;
    font-size: 0.8em;
    color: #999;
    text-align: right;
    font-style: italic;
}

.note-comments { border-left-color: #0984e3; }
.note-easier { border-left-color: #00b894; }
.note-harder { border-left-color: #d63031; }

.force-badge {
    display: inline-block;
    padding: 8px 16px;
    border-radius: 25px;
    font-weight: 600;
    margin: 5px;
    animation: float 3s ease-in-out infinite;
}

@keyframes float {
    0% { transform: translateY(0px); }
    50% { transform: translateY(-5px); }
    100% { transform: translateY(0px); }
}

.force-push { background: linear-gradient(135deg, #ff6b6b 0%, #ee5253 100%); color: white; }
.force-pull { background: linear-gradient(135deg, #48dbfb 0%, #0abde3 100%); color: white; }
.force-push-pull { background: linear-gradient(135deg, #feca57 0%, #ff9f43 100%); color: white; }

.stats-container {
    display: flex;
    justify-content: space-around;
    margin: 20px 0;
    padding: 15px;
    background: linear-gradient(135deg, #f6f9fc 0%, #e6f0f5 100%);
    border-radius: 15px;
}

.stat-item {
    text-align: center;
}

.stat-value {
    font-size: 1.5em;
    font-weight: 700;
    color: #667eea;
}

.stat-label {
    font-size: 0.8em;
    color: #666;
    text-transform: uppercase;
}

.loading-bar {
    width: 100%;
    height: 4px;
    background: linear-gradient(90deg, #667eea, #764ba2, #667eea);
    background-size: 200% 100%;
    animation: loading 2s infinite;
    border-radius: 2px;
    margin: 10px 0;
}

@keyframes loading {
    0% { background-position: 0% 50%; }
    100% { background-position: 200% 50%; }
}

.url-link {
    color: #667eea;
    text-decoration: none;
    font-size: 0.9em;
    word-break: break-all;
    transition: color 0.2s;
}

.url-link:hover {
    color: #764ba2;
    text-decoration: underline;
}
</style>
"""

# 3. Function to create HTML for an exercise
def create_exercise_html(exercise, index):
    # Get emoji for note type
    def get_note_emoji(note_type):
        emojis = {
            'comments': '',
            'easier': '',
            'harder': ''
        }
        return emojis.get(note_type, '')

    html = f"""
    <div class="exercise-container">
        <div class="exercise-card">
            <div class="exercise-header">
                <div class="exercise-number">{index}</div>
                <div class="exercise-title">{exercise.get('exercise_name', 'Unknown')}</div>
            </div>

            <div class="info-grid">
                <div class="info-item">
                    <span class="info-label"> Hierarchy</span>
                    <span class="info-value">{exercise.get('breadcrumb_path', 'N/A')}</span>
                </div>
                <div class="info-item">
                    <span class="info-label"> Source</span>
                    <a href="{exercise.get('source_url', '#')}" class="url-link" target="_blank">{exercise.get('source_url', 'N/A')[:50]}...</a>
                </div>
            </div>

            <div class="badge badge-primary"> Utility: {exercise.get('utility', 'N/A')}</div>
            <div class="badge badge-success"> Mechanics: {exercise.get('mechanics', 'N/A')}</div>
    """

    # Add force badge with special styling
    force = exercise.get('force', 'N/A')
    force_class = "force-push" if "push" in force.lower() else "force-pull" if "pull" in force.lower() else "force-push-pull"
    html += f'<div class="force-badge {force_class}"> Force: {force}</div>'

    # Muscles section
    if 'muscles' in exercise:
        html += '<div style="margin-top: 20px;">'
        for muscle_group, muscle_list in exercise['muscles'].items():
            formatted_group = muscle_group.replace('_', ' ').title()
            html += f"""
            <div class="muscle-group">
                <div class="muscle-group-title"> {formatted_group}</div>
                <div class="muscle-tags">
            """
            for muscle in muscle_list[:5]:  # Show top 5 muscles
                html += f'<span class="muscle-tag">{muscle}</span>'
            if len(muscle_list) > 5:
                html += f'<span class="muscle-tag">+{len(muscle_list)-5} more</span>'
            html += "</div></div>"
        html += "</div>"

    # Instructions section
    if 'instructions' in exercise:
        html += '<div style="margin-top: 20px;"><h3 style="color: #667eea; margin-bottom: 15px;">📝 Instructions</h3>'
        for step, text in exercise['instructions'].items():
            short_text = text[:150] + "..." if len(text) > 150 else text
            html += f"""
            <div class="instruction-step">
                <div class="instruction-title">{step}</div>
                <div class="instruction-text">{short_text}</div>
            </div>
            """
        html += "</div>"

    # Enhanced Extra Notes section with full content
    extras = ['comments', 'easier', 'harder']
    found_extras = [e for e in extras if e in exercise and exercise[e]]

    if found_extras:
        html += '<div class="extra-notes-section">'
        html += '<div class="extra-notes-title">Extra Notes & Variations</div>'

        for extra in found_extras:
            content = exercise[extra]
            word_count = len(content.split())

            # Determine color scheme based on note type
            note_class = f"note-{extra}"
            title_class = f"note-title {extra}"

            # Format the content with proper paragraphs
            formatted_content = content.replace('\n', '<br>')

            html += f"""
            <div class="extra-note-card {note_class}">
                <div class="note-header">
                    <div class="{title_class}">
                        {get_note_emoji(extra)} {extra.title()}
                    </div>
                    <span class="note-badge">{word_count} words</span>
                </div>
                <div class="note-content">
                    {formatted_content}
                </div>
                <div class="note-footer">
                    {word_count} words • Click to expand
                </div>
            </div>
            """
        html += "</div>"

    # Statistics
    html += f"""
    <div class="stats-container">
        <div class="stat-item">
            <div class="stat-value">{len(exercise.get('muscles', {}).get('target', []))}</div>
            <div class="stat-label">Target Muscles</div>
        </div>
        <div class="stat-item">
            <div class="stat-value">{len(exercise.get('instructions', {}))}</div>
            <div class="stat-label">Instruction Steps</div>
        </div>
        <div class="stat-item">
            <div class="stat-value">{len(found_extras)}</div>
            <div class="stat-label">Extra Notes</div>
        </div>
    </div>

    <div class="loading-bar"></div>
    </div></div>
    """

    return html

# 4. Display exercises with enhanced styling
display(HTML(css_styles))

# 5. Pick 3 random samples
samples = random.sample(database, min(3, len(database)))

# 6. Display each exercise with beautiful formatting
for i, exercise in enumerate(samples, 1):
    html_content = create_exercise_html(exercise, i)
    display(HTML(html_content))

    # Also print to console for comparison
    print("="*60)
    print(f" 🏋️‍♂️ SAMPLE {i}: {exercise.get('exercise_name', 'Unknown').upper()}")
    print("="*60)

    # Print extra notes to console
    extras = ['comments', 'easier', 'harder']
    found_extras = [e for e in extras if e in exercise]
    if found_extras:
        print("\n EXTRA NOTES:")
        for extra in found_extras:
            word_count = len(exercise[extra].split())
            print(f"   • {extra.title()}: {word_count} words")
            # Show preview
            preview = exercise[extra][:100] + "..." if len(exercise[extra]) > 100 else exercise[extra]
            print(f"     Preview: {preview}\n")
    print("\n")

# 7. Add a summary
total_exercises = len(database)
total_muscles = sum(len(ex.get('muscles', {}).get('target', [])) for ex in database)
total_instructions = sum(len(ex.get('instructions', {})) for ex in database)
total_extra_notes = sum(1 for ex in database for e in ['comments', 'easier', 'harder'] if e in ex)

summary_html = f"""
<div style="
    font-family: 'Inter', sans-serif;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 20px;
    border-radius: 15px;
    color: white;
    margin-top: 30px;
    text-align: center;
    animation: fadeIn 1s ease-out;
">
    <h2 style="margin-bottom: 15px;"> Database Summary</h2>
    <div style="display: flex; justify-content: space-around; flex-wrap: wrap;">
        <div>
            <div style="font-size: 2.5em; font-weight: bold;">{total_exercises}</div>
            <div>Total Exercises</div>
        </div>
        <div>
            <div style="font-size: 2.5em; font-weight: bold;">{total_muscles}</div>
            <div>Target Muscles</div>
        </div>
        <div>
            <div style="font-size: 2.5em; font-weight: bold;">{total_instructions}</div>
            <div>Instruction Steps</div>
        </div>
        <div>
            <div style="font-size: 2.5em; font-weight: bold;">{total_extra_notes}</div>
            <div>Extra Notes</div>
        </div>
    </div>
</div>
"""

display(HTML(summary_html))

 Successfully loaded 873 exercises from database.



 🏋️‍♂️ SAMPLE 1: LEVER ISOLATERAL CURL (PLATE LOADED)

 EXTRA NOTES:
   • Comments: 47 words
     Preview: The biceps may be exercised alternating or simultaneous. If machine does not provide support for bac...





 🏋️‍♂️ SAMPLE 2: WEIGHTED DECLINE SIT-UP

 EXTRA NOTES:
   • Comments: 68 words
     Preview: Exercise can be performed without added weight until more resistance is needed. Lower decline to inc...





 🏋️‍♂️ SAMPLE 3: KETTLEBELL TWO ARM SWING

 EXTRA NOTES:
   • Comments: 52 words
     Preview: Keep arms straight and back taut throughout swing. Dynamic shoulder and shoulder girdle movements ar...





In [ ]:
import pandas as pd
import json

# ==========================================
# 1. LOAD THE DATABASE
# ==========================================
file_path = '/content/final_exrx_database.json'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        database = json.load(f)
    df = pd.DataFrame(database)
except FileNotFoundError:
    print("❌ Could not find the JSON file. Check the path!")
    exit()

print("="*50)
print(" 🕵️‍♂️ DATA QUALITY AUDIT REPORT")
print("="*50)
print(f"Total Records: {len(df)}")
print(f"Total Columns: {len(df.columns)}\n")

# ==========================================
# 2. CHECK FOR DUPLICATES
# ==========================================
print("--- DUPLICATES ---")
# Checking if the exact same URL was scraped more than once
duplicate_urls = df.duplicated(subset=['source_url']).sum()
# Checking if there are any perfectly identical rows, excluding unhashable columns
exact_duplicates = df.drop(columns=['instructions', 'muscles'], errors='ignore').duplicated().sum()

print(f"Duplicate URLs found:  {duplicate_urls}")
print(f"Exact Duplicate rows:  {exact_duplicates}\n")

# ==========================================
# 3. CHECK FOR MISSING VALUES
# ==========================================
print("--- MISSING DATA ---")
# Calculate the total missing and the percentage missing for each column
missing_data = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

# Combine into a new DataFrame for a clean display
missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing %': missing_percent.round(2)
})

# Sort so the columns with the most missing data are at the top
missing_df = missing_df.sort_values(by='Missing %', ascending=False)

# Filter out columns that have 0 missing values for a cleaner report
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("Incredible! Your dataset has 0 missing top-level values.")
else:
    print(missing_df.to_string())

print("\n" + "="*50)

 🕵️‍♂️ DATA QUALITY AUDIT REPORT
Total Records: 873
Total Columns: 15

--- DUPLICATES ---
Duplicate URLs found:  34
Exact Duplicate rows:  0

--- MISSING DATA ---
              Missing Count  Missing %
force4                  872      99.89
function                799      91.52
intensity               799      91.52
easier                  706      80.87
harder                  706      80.87
muscles                 279      31.96
force                   131      15.01
mechanics               130      14.89
utility                 130      14.89
instructions             30       3.44
comments                 30       3.44

